# 03 — Desenvolvimento de Modelos
### Tech Challenge — Fase 2 | Pós Tech Data Analytics (14DTAT) | Luis Guilherme Betarelli Lopes (RM374939)

Este notebook carrega os dados já pré-processados (`02_preprocessamento.ipynb`)
e treina três modelos de classificação distintos, para depois comparar seus
desempenhos no notebook `04_avaliacao.ipynb`.


In [1]:
import pandas as pd
import joblib
import os

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
os.makedirs("../data/processed/models", exist_ok=True)

## 1. Carregando os dados processados

Os dados abaixo já saíram do notebook 02 com: `Id` removido, features novas
(`acid_ratio`, `sulfur_dioxide_ratio`) criadas, split estratificado 70/30
aplicado e padronização (`StandardScaler`, ajustado só no treino) já feita.
Nenhum novo cálculo de estatística é feito aqui em cima do conjunto de teste —
isso evita vazamento de dados entre as etapas.

In [2]:
train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")

X_train = train.drop(columns=["quality_label"])
y_train = train["quality_label"]
X_test = test.drop(columns=["quality_label"])
y_test = test["quality_label"]

print("Treino:", X_train.shape, "| Teste:", X_test.shape)

Treino: (800, 13) | Teste: (343, 13)


## 2. Escolha e treino dos modelos

Treinamos três modelos de classificação com abordagens bem diferentes entre
si, para poder comparar resultados:

- **KNN (K-Nearest Neighbors)** — classifica comparando cada vinho aos vizinhos
  mais parecidos já vistos no treino. Sensível a escala (por isso os dados já
  vêm padronizados do notebook 02).
- **Árvore de Decisão** — aprende uma sequência de regras do tipo "se-então"
  sobre as variáveis.
- **Random Forest** — combina centenas de árvores de decisão diferentes e
  agrega os "votos" para decidir, geralmente mais robusto que uma árvore
  isolada.

Como a EDA (notebook 01) mostrou que a base é desbalanceada (~14% de alta
qualidade), usamos `class_weight="balanced"` nos dois modelos baseados em
árvore — isso ajusta o peso das classes durante o treino para compensar a
proporção desigual. O KNN não tem esse parâmetro; seu comportamento em relação
ao desbalanceamento é discutido na avaliação (notebook 04).

`RANDOM_STATE = 42` é usado em todos os pontos com componente aleatória, para
garantir que o experimento seja reprodutível.

In [3]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

tree_clf = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced", max_depth=6)
tree_clf.fit(X_train, y_train)

rf_clf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced", n_estimators=300)
rf_clf.fit(X_train, y_train)

modelos = {
    "KNN": knn,
    "Árvore de Decisão": tree_clf,
    "Random Forest": rf_clf,
}
print("Modelos treinados:", list(modelos.keys()))

Modelos treinados: ['KNN', 'Árvore de Decisão', 'Random Forest']


## 3. Salvando os modelos treinados

In [4]:
for nome, modelo in modelos.items():
    filename = nome.lower().replace(" ", "_").replace("á", "a").replace("ã", "a")
    joblib.dump(modelo, f"../data/processed/models/{filename}.pkl")

print("Modelos salvos em data/processed/models/")

Modelos salvos em data/processed/models/


## Resumo para o próximo notebook

- Três modelos treinados com os mesmos dados de treino: KNN, Árvore de Decisão
  e Random Forest.
- Modelos salvos em `data/processed/models/*.pkl` (não versionados no Git —
  são reproduzíveis a partir deste notebook).
- Próximo passo (`04_avaliacao.ipynb`): avaliar os três modelos no conjunto de
  teste com métricas além de acurácia, e interpretar os resultados.